In [ ]:
!pip install enoslib ipywidgets fabric --break-system-packages

In [ ]:
!ssh rennes.grid5000.fr hostname

### G5K connection

Setup the `.python-grid5000.yaml` file with the username and password used to login to grid5000.

The file should look like this:
```yaml
username: G5K_LOGIN
password: G5K_password
```

-> required in order for the enoslib calls to g5k to work.

In addition to this, make sure to add these lines to your ssh configuration:

```text
Host g5k
    User G5K_LOGIN
    HostName access.grid5000.fr
    ForwardAgent no

Host !access.grid5000.fr *.grid5000.fr
    User G5K_LOGIN
    ProxyJump G5K_LOGIN@access.grid5000.fr
    StrictHostKeyChecking no
    UserKnownHostsFile /dev/null
    ForwardAgent yes

Host access.grid5000.fr
    User G5K_LOGIN
    StrictHostKeyChecking no
    UserKnownHostsFile /dev/null
    ForwardAgent yes

Host *.g5k
    User G5K_LOGIN
    ProxyCommand ssh g5k -W "$(basename %h .g5k):%p"
    ForwardAgent no
```

Make sure to replace `G5K_LOGIN` with your g5k username (the one from the site)

### Job configuration

Setup the various parameters for the job:
- The job name will identify the current booking, if the notebook kernel dies, re running the same reservation code with the same name will reload the existing job instead of booking a new one
- The walltime is the time that the booking will last, you can always stop your reservation earlier than the booking's end time

In [ ]:
import os
from grid5000 import Grid5000
import enoslib as en
import logging
from datetime import datetime, timedelta

conf_file = os.path.join(os.environ.get("HOME"), ".python-grid5000.yaml")  # type: ignore
gk = Grid5000.from_yaml(conf_file)

en.set_config(ansible_forks=100)

# map each cluster to its site
cluster_to_site = {}
for site in gk.sites.list():
    for cluster in site.clusters.list():
        cluster_to_site[cluster.uid] = site.uid

# JOB CONFIGURATION
JOB_NAME = "fcquic_relay_eval_multisite"
JOB_WALLTIME = timedelta(hours=2, minutes=0)

# usage policy check: the job cannot cross the day to night boundary at 7pm if it was submitted before 5 pm. Any job started after 5 pm can cross the boundary
# I had the issue once so this check is there to avoid receiving a usage policy violation email...
datetime_now = datetime.now()
job_end_dt = datetime_now + JOB_WALLTIME
if datetime_now.hour <= 17 and job_end_dt.hour >= 19:
    raise RuntimeError(
        "This job reservation will violate the usage policy and will cross the day night boundary"
    )


# --------------------- NOTE: very important !!!! changing the server cluster and number below will do nothing, the server is set to chirop5 right now
NUM_SERVER_NODES = 1
SERVER_CLUSTER = "chirop"  # Lille
# ----------------------


# WARNING: cluster dahu doesn't work with multicast

# number of network namespaces per client server (each ns runs one client binary)
# per discussion with the prof. 5 to 10 namespaces per server is fine
NUM_NS_PER_CLIENT = 5


# LARGE TOPO
CLIENT_CLUSTERS = [
    # cluster 0
    {"cluster": "gros", "num_clients": 5},  # nancy
    # cluster 1
    {"cluster": "parasilo", "num_clients": 5},  # rennes
    # {"cluster": "paradoxe", "num_clients": 5},  # rennes
    # cluster 2
    {"cluster": "ecotype", "num_clients": 5},  # nantes
    # {"cluster": "econome", "num_clients": 5},  # nantes
    # cluster 3
    {"cluster": "nova", "num_clients": 5},  # lyon
]

# each entry in the table here is a link between two routers.
# role names are the keys: "router_server", "router_client_0",...
TOPOLOGY_LINKS = [
    ("router_server", "router_client_0"),  # src to nancy
    ("router_client_0", "router_client_1"),  # nancy to rennes
    # ("router_client_0", "router_client_2"),  # nancy to lyon
    ("router_client_0", "router_client_3"),  # nancy to lyon
    ("router_client_1", "router_client_2"),  #  rennes to nantes
]

# small topo
# CLIENT_CLUSTERS = [
#     # cluster 0
#     {"cluster": "paradoxe", "num_clients": 3},  # rennes
#     # cluster 1
#     {"cluster": "ecotype", "num_clients": 5},  # nantes
# ]

# # each entry in the table here is a link between two routers.
# # role names are the keys: "router_server", "router_client_0",...
# TOPOLOGY_LINKS = [
#     ("router_server", "router_client_0"),  # src to rennes
#     ("router_client_0", "router_client_1"),  # rennes to nantes
# ]

# LARGE TOPO
# CLIENT_CLUSTERS = [
#     # cluster 0
#     {"cluster": "paradoxe", "num_clients": 3},  # rennes
#     # cluster 1
#     {"cluster": "nova", "num_clients": 5},  # lyon
#     # cluster 2
#     {"cluster": "ecotype", "num_clients": 5},  # nantes
#     # cluster 3
# ]

# # each entry in the table here is a link between two routers.
# # role names are the keys: "router_server", "router_client_0",...
# TOPOLOGY_LINKS = [
#     ("router_server", "router_client_0"), # src to rennes
#     ("router_server", "router_client_1"), # src to lyon
#     ("router_client_0", "router_client_2"), # rennes to nantes
#     ("router_client_1", "router_client_3"), # lyon to grenoble
# ]

# CLIENT_CLUSTERS = [
#     # cluster 0
#     {"cluster": "ecotype", "num_clients": 1},  # nantes
# ]
# TOPOLOGY_LINKS = [
#     ("router_server", "router_client_0"),  # rennes -> nantes
# ]

# Display some general information about the library
en.check()
# Enable rich logging
_ = en.init_logging()
# en.set_config(g5k_auto_jump=False)

conf = (
    en.G5kConf.from_settings(
        job_name=JOB_NAME,
        walltime=str(JOB_WALLTIME),
        env_name="debian12-nfs",
        job_type=["deploy"],
    )
    # server router
    .add_machine(
        roles=["router", "router_server"],
        cluster=SERVER_CLUSTER,
        nodes=1,
    )
    # server
    .add_machine(
        roles=["server"],
        servers=["chirop-5.lille.grid5000.fr"],
        # cluster=SERVER_CLUSTER,
        # nodes=NUM_SERVER_NODES,
    ).add_network(
        id="subnet_server",
        type="slash_22",
        roles=["subnet", "subnet_server"],
        site=cluster_to_site[SERVER_CLUSTER],
    )
)

# we need to add one client router + clients + relay + subnet for each client cluster
for i, client_cluster in enumerate(CLIENT_CLUSTERS):
    conf = (
        conf
        # add only one client router
        .add_machine(
            roles=["router", "router_client", f"router_client_{i}"],
            cluster=client_cluster["cluster"],
            nodes=1,
        )
        # add all of the client machines
        .add_machine(
            roles=["client", f"client_{i}"],
            cluster=client_cluster["cluster"],
            nodes=client_cluster["num_clients"],
        )
        # one relay per client cluster
        .add_machine(
            roles=["relay", f"relay_{i}"],
            cluster=client_cluster["cluster"],
            nodes=1,
        ).add_network(
            id=f"subnet_client_{i}",
            type="slash_22",
            roles=["subnet", "subnet_client", f"subnet_client_{i}"],
            site=cluster_to_site[client_cluster["cluster"]],
        )
    )

# This will validate the configuration, but not reserve resources yet
provider = en.G5k(conf)

In [ ]:
print("Reserving resources...")

# Get actual resources
roles, networks = provider.init()
display(roles)
display(networks)

# Fill in network information from nodes
roles = en.sync_info(roles, networks)

with en.actions(roles=roles) as a:
    a.apt(task_name="Install traceroute", name="traceroute", state="present")
    a.apt(task_name="Install btop", name="btop", state="present")
    a.apt(task_name="Install htop", name="htop", state="present")
    a.apt(task_name="Install tcpdump", name="tcpdump", state="present")
    a.apt(
        task_name="Install python",
        name=["python3-pip", "python-is-python3"],
        state="present",
    )

with en.actions(roles=roles["router"], gather_facts=True) as a:
    a.file(
        task_name="Ensure apt keyring directory exists",
        path="/usr/share/keyrings",
        state="directory",
        mode="0755",
    )
    a.get_url(
        task_name="Download FRR GPG key",
        url="https://deb.frrouting.org/frr/keys.gpg",
        dest="/usr/share/keyrings/frrouting.gpg",
        mode="0644",
    )
    a.apt_repository(
        task_name="Add FRR apt repository",
        repo="deb [signed-by=/usr/share/keyrings/frrouting.gpg] https://deb.frrouting.org/frr {{ ansible_distribution_release }} frr-stable",
        filename="frr",
        state="present",
    )
    a.apt(
        task_name="Install FRR packages",
        name=["frr", "frr-pythontools"],
        state="present",
        update_cache=True,
    )
    results = a.results

### Interface setup
- We first need to get the name of the primary interface for each node, this is the interface that is connected to the "prod" network
- Then we need to assign IPs from our subnet to the nodes


In [ ]:
from ipaddress import ip_address, ip_network

prod_interfaces_per_node = {}

# find the physical interface connected to the production network

for host in roles["client"] + roles["server"] + roles["router"] + roles["relay"]:
    node_name = host.address

    prod_interfaces = host.filter_interfaces(networks=networks["prod"])
    if prod_interfaces:
        prod_interface_name = prod_interfaces[0]
        print(f"Prod interface for {host.alias}: {prod_interface_name}")
        prod_interfaces_per_node[host.alias] = prod_interface_name

    else:
        print(
            f"Couldn't find prod iface for {host.alias}, checking each interface directly"
        )
        prod_network = ip_network("172.0.0.0/8")
        for interface in host.net_devices:
            for address in interface.addresses:
                if address.ip in prod_network:
                    prod_interfaces_per_node[host.alias] = interface.name


display(prod_interfaces_per_node)

subnet_cluster_mapping = {}
for i, client_cluster in enumerate(CLIENT_CLUSTERS):
    site = client_cluster["cluster"]
    subnet = networks[f"subnet_client_{i}"][0].network
    subnet_cluster_mapping[site] = str(subnet.network_address)

display(subnet_cluster_mapping)

### Assigning IPs from subnets

In [ ]:
from itertools import islice, product
import subprocess

node_ips = {}
# all namespace IPs across all client nodes (flat list for passing to NPF)
all_ns_ips = []

server_ips = networks["subnet_server"][0].free_ips


def assign_n_ips_to_hosts(role, N, ips):
    global node_ips

    for host in roles[role]:

        host_prod_iface = prod_interfaces_per_node[host.alias]
        host.extra.update(ips=[str(ip) for ip in islice(ips, N)])

        for ip in host.extra.get("ips"):

            print(f"Adding ip {ip} to host: {host.alias}")

            if node_ips.get(role) is None:
                node_ips[role] = []
            node_ips[role].append(ip)

            if "router" not in role:
                cmd = f"(ip a | grep {ip}) || ip addr add {ip}/32 dev {host_prod_iface}"
                en.run_command(cmd, task_name="cmd", roles=host, gather_facts=False)

        if "router" in role:

            # get each node's IP address on the production network
            ip_address_list = host.filter_addresses(networks=networks["prod"])
            if len(ip_address_list) > 0:
                ip_address_obj = ip_address_list[0]
            else:
                # if we cant obtain info from the host's production network (it's buggy in LLN), then fetch the ip directly
                prod_network = ip_network("172.0.0.0/8")
                for interface in host.net_devices:
                    for address in interface.addresses:
                        if address.ip in prod_network:
                            ip_address_obj = address

            # This may seem weird: ip_address_obj.ip is a `netaddr.IPv4Interface`
            # which itself has an `ip` attribute.
            node_ip = ip_address_obj.ip.ip
            if node_ips.get(role) is None:
                node_ips[role] = []
            node_ips[role].append(node_ip.exploded)
            host.extra.update(ips=node_ips[role])


def assign_ns_ips_to_clients(role, ips):
    # assign ip addresses of the client nodes
    global node_ips, all_ns_ips

    for host in roles[role]:
        ns_ips = [str(ip) for ip in islice(ips, NUM_NS_PER_CLIENT)]
        host.extra.update(ips=ns_ips)
        host.extra.update(
            ns_configs=[{"id": j, "ip": ns_ips[j]} for j in range(len(ns_ips))]
        )

        if node_ips.get(role) is None:
            node_ips[role] = []
        node_ips[role].extend(ns_ips)
        all_ns_ips.extend(ns_ips)

        print(
            f"Allocated {len(ns_ips)} namespace IP addresses for {host.alias}: {ns_ips}"
        )


assign_n_ips_to_hosts("router_server", 1, server_ips)
assign_n_ips_to_hosts("server", 1, server_ips)

# assign ips to all clients, relays, and routers in each cluster
for i, client_cluster in enumerate(CLIENT_CLUSTERS):
    client_ips = networks[f"subnet_client_{i}"][0].free_ips
    assign_n_ips_to_hosts(f"router_client_{i}", 1, client_ips)
    # allocate NUM_NS_PER_CLIENT IPs per client node (instead of 1)
    assign_ns_ips_to_clients(f"client_{i}", client_ips)
    # one IP for the relay in this cluster
    assign_n_ips_to_hosts(f"relay_{i}", 1, client_ips)


display(node_ips)
print(f"number of total ip addresses (i.e., indiviual client): {len(all_ns_ips)}")
display(all_ns_ips)

### Client network namespace setup (MACVLAN)

Each client node gets `NUM_NS_PER_CLIENT` network namespaces connected to the prod interface via a MACVLAN "bridge", which is not really a bridge. Using bridges and virtual ethernet would probs work but it's such a mess

In [ ]:
from ipaddress import ip_address, ip_network

all_client_roles = [f"client_{i}" for i in range(len(CLIENT_CLUSTERS))]

# creating the namespaces:
# we need the gateway IP per cluster for the default routes of the namespaces
# the router's subnet IP (10.xzy) is in the same /22 subnet as the namespace IPS
# so we use that as the gateway (the global/prod IP is on a different subnet).
gateway_ip_per_cluster = {}
for i in range(len(CLIENT_CLUSTERS)):
    router_role = f"router_client_{i}"
    local_subnet = ip_network("10.0.0.0/8")
    host_ips = [
        str(ip) for ip in node_ips[router_role] if ip_address(ip) in local_subnet
    ]
    if not host_ips:
        raise RuntimeError(f"No subnet IP for {router_role}")
    gateway_ip_per_cluster[i] = host_ips[0]

for i, client_cluster in enumerate(CLIENT_CLUSTERS):
    role = f"client_{i}"
    gateway_ip = gateway_ip_per_cluster[i]
    print(f"gateway_ip={gateway_ip} for client {role}")

    for host in roles[role]:
        prod_iface = prod_interfaces_per_node[host.alias]
        # store prod_iface and gateway in extra so we can use them in the jinja template of en.play_on (see enoslib docs on ansible)
        host.extra.update(prod_iface=prod_iface, ns_gateway=gateway_ip)

    with en.play_on(roles=roles, pattern_hosts=role, gather_facts=False) as p:
        # NOTE: the commands below will run as root iif the ssh keys setup in g5k are present on the current machine
        p.shell(
            """
            NS_NAME="client-{{ item.id }}"
            MACVLAN_HOST="mv-c{{ item.id }}"
            MACVLAN_NS="eth0"
            IP_ADDR="{{ item.ip }}"
            PROD_IFACE="{{ prod_iface }}"
            GATEWAY="{{ ns_gateway }}"

            ip netns add "$NS_NAME"

            # create a MACVLAN interface on the prod interface
            ip link add "$MACVLAN_HOST" link "$PROD_IFACE" type macvlan mode bridge
            ip link set "$MACVLAN_HOST" netns "$NS_NAME"

            # configure the netns interface
            ip netns exec "$NS_NAME" ip link set "$MACVLAN_HOST" name "$MACVLAN_NS"
            ip netns exec "$NS_NAME" ip addr add "$IP_ADDR"/22 dev "$MACVLAN_NS"
            ip netns exec "$NS_NAME" ip link set "$MACVLAN_NS" up
            ip netns exec "$NS_NAME" ip link set lo up
            ip netns exec "$NS_NAME" ip link set "$MACVLAN_NS" multicast on
            ip netns exec "$NS_NAME" ip route add default via "$GATEWAY" dev "$MACVLAN_NS"

            sysctl -w net.core.rmem_max=26214400
            sysctl -w net.core.rmem_default=26214400
            """,
            loop="{{ ns_configs }}",
            task_name="create_macvlan_namespaces",
        )

    print(
        f"Created {len(roles[role]) * NUM_NS_PER_CLIENT} namespaces for {role} (with gateway {gateway_ip})"
    )

### GRE Tunnels setup


In [ ]:
from ipaddress import ip_network, ip_address
from collections import defaultdict


def get_global_ip_for_role(host, role: str) -> str:
    local_subnet = ip_network(
        "10.0.0.0/8"
    )  # the subnets we get are in 10..../8, can't be more specific than that sadly
    # since we fetched the global address for the routers and assigned them a local address
    # and we want the global address, we filter out the local address
    host_ips = [
        str(ip)
        for ip in host.extra.get("ips", [])
        if ip_address(ip) not in local_subnet
    ]
    if host_ips:
        return host_ips[0]
    raise RuntimeError(f"Could get prod ip for '{host.address}'")


# per router tunnel info: router_tunnels[role] -> list of {iface, ip, network, tunnel_subnet}
# used in frr config cell
router_tunnels = defaultdict(list)

# per router GRE interface counter (each router gets gre1, gre2, ... for each link it participates in)
gre_counter = defaultdict(int)

# per router GRE shell commands
# NOTE: we execute everthing at the same time otherwise there can be issues with reachability
router_gre_cmds = defaultdict(list)

tunnel_base = int(ip_address("192.168.0.0"))

for link_idx, (role_a, role_b) in enumerate(TOPOLOGY_LINKS):
    host_a = roles[role_a][0]
    host_b = roles[role_b][0]

    prod_ip_a = get_global_ip_for_role(host_a, role_a)
    prod_ip_b = get_global_ip_for_role(host_b, role_b)

    # /30 tunnel subnet for this link
    tunnel_subnet = ip_network((tunnel_base + link_idx * 4, 30))
    tunnel_ip_a = str(tunnel_subnet.network_address + 1)
    tunnel_ip_b = str(tunnel_subnet.network_address + 2)

    # GRE interface names
    gre_counter[role_a] += 1
    gre_counter[role_b] += 1
    gre_iface_a = f"gre{gre_counter[role_a]}"
    gre_iface_b = f"gre{gre_counter[role_b]}"

    # tunnel metadata used in FRR config
    router_tunnels[role_a].append(
        {
            "iface": gre_iface_a,
            "ip": tunnel_ip_a,
            "network": str(tunnel_subnet.network_address),
            "tunnel_subnet": tunnel_subnet,
        }
    )
    router_tunnels[role_b].append(
        {
            "iface": gre_iface_b,
            "ip": tunnel_ip_b,
            "network": str(tunnel_subnet.network_address),
            "tunnel_subnet": tunnel_subnet,
        }
    )

    # GRE commands for side A
    router_gre_cmds[role_a].extend(
        [
            f"sudo ip link del {gre_iface_a} 2>/dev/null || true",
            f"sudo ip tunnel add {gre_iface_a} mode gre local {prod_ip_a} remote {prod_ip_b} ttl 255",
            f"sudo ip addr add {tunnel_ip_a}/30 dev {gre_iface_a}",
            f"sudo ip link set {gre_iface_a} up",
            f"sudo ip link set {gre_iface_a} multicast on",
            f"sudo sysctl -w net.ipv4.conf.{gre_iface_a}.rp_filter=0",
            "sudo sysctl -w net.ipv4.conf.all.rp_filter=0",
        ]
    )

    # GRE commands for side B
    router_gre_cmds[role_b].extend(
        [
            f"sudo ip link del {gre_iface_b} 2>/dev/null || true",
            f"sudo ip tunnel add {gre_iface_b} mode gre local {prod_ip_b} remote {prod_ip_a} ttl 255",
            f"sudo ip addr add {tunnel_ip_b}/30 dev {gre_iface_b}",
            f"sudo ip link set {gre_iface_b} up",
            f"sudo ip link set {gre_iface_b} multicast on",
            f"sudo sysctl -w net.ipv4.conf.{gre_iface_b}.rp_filter=0",
            "sudo sysctl -w net.ipv4.conf.all.rp_filter=0",
        ]
    )

    print(
        f"Link {link_idx}: {gre_iface_a}({role_a}, {tunnel_ip_a}) <-> {gre_iface_b}({role_b}, {tunnel_ip_b})"
    )

for role, cmds in router_gre_cmds.items():
    host = roles[role][0]
    en.run_command(
        "; ".join(cmds), task_name=f"setup_gre_{role}", roles=host, gather_facts=False
    )
    print(f"Created {len(router_tunnels[role])} GRE tunnels on {role} ({host.address})")

display(dict(router_tunnels))

### FRR Routing setup

In [ ]:
from ipaddress import ip_address, ip_network
from pathlib import Path
import subprocess
from jinja2 import Template

# router mapping: (role, subnet_key) for every router is built dynamically
ROUTER_MAPPING: list[tuple[str, str]] = [
    ("router_server", "subnet_server"),
]
for i in range(len(CLIENT_CLUSTERS)):
    ROUTER_MAPPING.append((f"router_client_{i}", f"subnet_client_{i}"))

TEMPLATE_FILE = "base_router_config_ospf.frr"
DAEMONS_FILE = "./daemons"
OUTPUT_DIR = Path("./generated_frr_configs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(TEMPLATE_FILE, "r", encoding="utf-8") as f:
    template = Template(f.read())


def parse_subnet(subnet_obj):
    for attr in ("network", "cidr", None):
        # try to get subnet_obj.attr (use the string repr of subnet_obj and parse it with ip_network if it's a string)
        # enoslib networks sutff is really annoying holy
        val = str(getattr(subnet_obj, attr, subnet_obj)) if attr else str(subnet_obj)
        if not val:
            continue
        try:
            return ip_network(val if "/" in val else f"{val}/22", strict=False)
        except ValueError:
            print(f"Failed to parse {val} ip network")
            continue
    raise RuntimeError(f"Can't parse subnet: {subnet_obj!r}")


def get_router_subnet_or_global_ip(host, role: str, get_subnet_ip: bool) -> str:
    local_subnet = ip_network(
        "10.0.0.0/8"
    )  # the subnets /22 we get from g5k are all in the 10..../8 subnet, can't be more specific than that sadly
    # since we fetched the global address for the routers and assigned them a local address

    host_ips = []
    for extra_ip in host.extra.get("ips", []):
        extra_ip_addr = ip_address(extra_ip)
        # if we want to fetch the local address, then only add the address if it is in the local subnet
        if extra_ip_addr in local_subnet and get_subnet_ip:
            host_ips.append(str(extra_ip))

        # if we want to fetch the global address, then only add the address if it NOT in the local subnet
        elif not get_subnet_ip and extra_ip_addr not in local_subnet:
            host_ips.append(str(extra_ip))

    if host_ips:
        return host_ips[0]

    role_ips = [str(ip) for ip in node_ips.get(role, [])]
    if role_ips:
        return role_ips[0]

    raise ValueError(f"Could get prod ip for '{host.address}'")


def get_default_gateway(address):
    # get the default gateway from the node via ssh
    # could just hardcode these values based on the info on the website...
    result = subprocess.run(
        ["ssh", address, "ip -4 route show default"],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0 or not result.stdout.strip():
        raise RuntimeError(f"No default route on {address}: {result.stderr.strip()}")

    out = result.stdout.strip().splitlines()[0].split()
    if "via" not in out:
        raise RuntimeError(f"No gateway used in the default route: {address}")

    return out[out.index("via") + 1]


def pick_loopback(subnet_obj, reserved):
    # since the booked subnets are /22, we get 3 different /24 subnets, so we just make sure that the routers have a loopback address in a
    # /24 subnet that we wont pick for the clients, just to be safe
    subnet = parse_subnet(subnet_obj)
    for ip_int in range(
        int(subnet.broadcast_address) - 1, int(subnet.network_address), -1
    ):
        candidate = str(ip_address(ip_int))
        if candidate not in reserved:
            return candidate
    raise ValueError(f"No free loopback in {subnet}")


reserved_ips = {str(ip) for ips in node_ips.values() for ip in ips}

for idx, (role, subnet_key) in enumerate(ROUTER_MAPPING):

    if not (subnet_key in networks and networks[subnet_key]):
        raise RuntimeError(f"Missing subnet {subnet_key}")

    host = roles[role][0]
    iface = prod_interfaces_per_node[host.address]
    subnet_obj = networks[subnet_key][0]

    prod_ip = get_router_subnet_or_global_ip(host, role, True)
    global_ip = get_router_subnet_or_global_ip(host, role, False)
    net_addr = str(parse_subnet(subnet_obj).network_address)
    lo_addr = pick_loopback(subnet_obj, reserved_ips)
    reserved_ips.add(lo_addr)
    gateway = get_default_gateway(host.address)

    router_id = idx + 1
    is_server = role == "router_server"

    # REMINDER: highest bsr priority wins
    # but lowest rp priority wins
    bsr_prio = router_id + 100 if is_server else router_id
    rp_prio = 0 if is_server else router_id + 100

    tunnels = [
        {"iface": t["iface"], "ip": t["ip"], "network": t["network"]}
        for t in router_tunnels.get(role, [])
    ]

    config = template.render(
        lo_address=lo_addr,
        prod_iface=iface,
        prod_net_ip=prod_ip,
        global_ip=global_ip,
        prod_network=net_addr,
        tunnels=tunnels,
        router_id=f"{router_id}.{router_id}.{router_id}.{router_id}",
        isis_router_id=router_id + 1,
        rp_prio=rp_prio,
        bsr_prio=bsr_prio,
        gateway=gateway,
    )

    # write locally and upload file to remote
    local_path = OUTPUT_DIR / f"{host.address.replace('/', '_')}.frr.conf"
    local_path.write_text(config, encoding="utf-8")

    remote = f"root@{host.address}:/etc/frr"
    subprocess.run(["scp", str(local_path), f"{remote}/frr.conf"], check=True)
    subprocess.run(["scp", DAEMONS_FILE, f"{remote}/daemons"], check=True)
    en.run_command(
        "sudo systemctl restart frr",
        task_name=f"restart_frr_{host.address}",
        roles=host,
        gather_facts=False,
    )

    print(
        f"[{role}] {host.address}  prod={prod_ip}  loopback={lo_addr}  gateway={gateway}  tunnels={len(tunnels)}"
    )

### Default route setup on the non router nodes

In [ ]:
import concurrent.futures
import subprocess

# figure out the gateway router for the server, each client, and each relay
gateway_router_role = {"server": "router_server"}
for i in range(len(CLIENT_CLUSTERS)):
    gateway_router_role[f"client_{i}"] = f"router_client_{i}"
    gateway_router_role[f"relay_{i}"] = f"router_client_{i}"


def get_global_ip_for_role(router_role) -> str:
    local_subnet = ip_network(
        "10.0.0.0/8"
    )  # the subnets we get are in 10..../8, can't be more specific than that sadly
    # since we fetched the global address for the routers and assigned them a local address
    # and we want the global address, we filter out the local address

    host_ips = [
        str(ip) for ip in node_ips[router_role] if ip_address(ip) not in local_subnet
    ]
    if host_ips:
        return host_ips[0]
    raise RuntimeError(f"Could get prod ip for {router_role}")


# Collect all tasks (host_alias, command, gateway_ip, iface) first, then run in parallel
route_tasks = []

for role, router_role in gateway_router_role.items():

    # skip any bad role written above
    if role not in roles or not roles[role]:
        print(f"Unknown role: {role}")
        continue

    gateway_ip = get_global_ip_for_role(router_role)
    print(gateway_ip)

    for host in roles[role]:
        host_iface = prod_interfaces_per_node.get(host.address)
        if host_iface is None:
            raise RuntimeError(f"Missing prod interface for node '{host.address}'")

        cmd = "; ".join(
            [
                f"sudo ip route replace default via {gateway_ip} dev {host_iface}",
                "sudo ip route flush cache",
                "ip route show default",
            ]
        )

        route_tasks.append((host.alias, cmd, gateway_ip, host_iface))


def set_default_route_on_host(host_alias, cmd, gateway_ip, host_iface):
    result = subprocess.run(
        ["ssh", host_alias, cmd],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        print(f"Error on {host_alias}: {result.stderr.strip()}")
    else:
        out = result.stdout.strip().splitlines()
        if out:
            print(out[0])
        print(f"{host_alias}'s default route is {gateway_ip} on {host_iface}")


print(f"setting default routes on {len(route_tasks)} nodes")
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
    pool.map(lambda t: set_default_route_on_host(*t), route_tasks)

### Uploading the binaries with rsync

In [ ]:
!cd /home/corentin/fcquic_applications_master_thesis/fcquic_relay && cargo build --release

In [ ]:
import concurrent.futures
import subprocess
import enoslib

relay_dir = "/home/corentin/fcquic_applications_master_thesis/fcquic_relay"
local_bin_dir = f"{relay_dir}/target/release"
remote_bin_dir = "/tmp/"

all_clients = [
    host for i in range(len(CLIENT_CLUSTERS)) for host in roles[f"client_{i}"]
]


def run_cmd(host, cmd):
    subprocess.run(["ssh", host, cmd], check=True)


def rsync_to_host(host, srcs, dest):
    subprocess.run(
        ["rsync", "-az", *srcs, f"{host}:{dest}"],
        check=True,
    )


def push_client_host(node):
    host = node.alias
    print(f"pushing binaries to {host}")
    run_cmd(
        host,
        f"mkdir -p {remote_bin_dir}/bin {remote_bin_dir}/logs/client {remote_bin_dir}/logs/server",
    )
    rsync_to_host(
        host,
        [f"{local_bin_dir}/server", f"{local_bin_dir}/client"],
        f"{remote_bin_dir}/bin/",
    )
    rsync_to_host(
        host, [f"{relay_dir}/cert.crt", f"{relay_dir}/cert.key"], remote_bin_dir
    )


def push_relay_host(node):
    host = node.alias
    print(f"pushing relay binaries to {host}")
    run_cmd(host, f"mkdir -p {remote_bin_dir}/bin {remote_bin_dir}/logs/relay")
    rsync_to_host(
        host,
        [f"{local_bin_dir}/fcquic_relay", f"{local_bin_dir}/app_relay"],
        f"{remote_bin_dir}/bin/",
    )
    rsync_to_host(
        host, [f"{relay_dir}/cert.crt", f"{relay_dir}/cert.key"], remote_bin_dir
    )


with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
    pool.map(push_client_host, all_clients + roles["server"])

relay_hosts = [
    node for i in range(len(CLIENT_CLUSTERS)) for node in roles[f"relay_{i}"]
]
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
    pool.map(push_relay_host, relay_hosts)

print("pushed binaries to all nodes")

res = en.run_command(
    "sysctl -w net.core.rmem_default=26214400 && sysctl -w net.core.rmem_max=26214400",
    roles=roles,
)
print(
    "errors: " + str([out.stderr for out in res.filter(status=enoslib.STATUS_FAILED)])
)

### Running the experiment

In [ ]:
raise RuntimeError("Don't launch this cell")

import datetime
from npf import enoslib as enoslib_npf
import npf.globals
from importlib import reload

reload(npf)

# IMPORTANT: if you run this cell multiple times, you must clear the global roles dictionary kept by NPF otherwise it will keep appending the nodes to it and this dict. will keep growing
# leading to each client being ran multiple times....
npf.globals.roles.clear()

# all_ns_ips was built in the IP assignment cell and contains every namespace IP
print(f"Total client IPs (namespaces): {len(all_ns_ips)}")

# one relay ip per cluster
relay_ips = [node_ips[f"relay_{i}"][0] for i in range(len(CLIENT_CLUSTERS))]
print(f"Relay IPs (one per cluster): {relay_ips}")


# create new roles for namespace clients client nodes need user="root" because sudo isn't available
# on the default user account, and NPF needs root to run commands in namespaces.
npf_roles = {}
for role, hosts in roles.items():
    if role == "client":
        # if not "client" in npf_roles or len(npf_roles["client"]) == 0:
        # npf_roles["client"] = []

        npf_roles["client"] = [
            en.Host(h.address, alias=h.alias, user="root", extra=h.extra) for h in hosts
        ]

    if role == "server":
        npf_roles[role] = hosts

    # # skip routers
    # elif role.startswith("router"):
    #     continue

    # else:
    #     npf_roles[role] = hosts

# # merge relay_0, relay_1,... into a single relay role so that NPF assigns
# # NPF_NODE_ID=0 to relay_0, NPF_NODE_ID=1 to relay_1...
npf_roles["relay"] = [
    h for i in range(len(CLIENT_CLUSTERS)) for h in roles[f"relay_{i}"]
]

display(npf_roles)

print("Launching NPF")

now = datetime.datetime.now().strftime("%d-%m-%H-%M%p")
test_name = f"large_relay_topo_{now}"

LATENCY_TEST = True
if not LATENCY_TEST:
    test_name = f"segmentation_test_{test_name}"
else:
    test_name = f"latency_test_{test_name}"

results, _ = enoslib_npf.run(
    "relay_eval.npf",
    argsv=[
        "--single-output",
        f"./npf-out/{test_name}.csv",
        "--no-graph",
        "--debug",
        *(["--tags", "data"] if not LATENCY_TEST else []),
        "--force-retest",
        f"--variables",
        f"SERVER_IP={node_ips['server'][0]}",
        f"SERVER_PROD_IFACE={prod_interfaces_per_node[roles["server"][0].alias]}",
        f"CLIENT_IPS=({' '.join(all_ns_ips)})",
        f"NUM_NS_PER_CLIENT={NUM_NS_PER_CLIENT}",
        f'RELAY_IPS={" ".join(relay_ips)}',
        # f"CLUSTER_NAMES={' '.join(subnet_cluster_mapping.keys())}",
        # f"CLUSTER_SUBNETS={' '.join(subnet_cluster_mapping.values())}",
    ],
    roles=npf_roles,
)

# CLEAN_CLUSTER_NAMES=""
# IFS=' ' read -ra CLUSTER_NAMES_ARR <<< "$CLUSTER_NAMES"
# for i in "${CLUSTER_NAMES_ARR[@]}"; do
#     CLEAN_CLUSTER_NAMES="${CLEAN_CLUSTER_NAMES} --cluster-names=${i}"
# done

# CLEAN_CLUSTER_SUBNETS=""
# IFS=' ' read -ra CLUSTER_SUBNETS_ARR <<< "$CLUSTER_SUBNETS"
# for i in "${CLUSTER_SUBNETS_ARR[@]}"; do
#     CLEAN_CLUSTER_SUBNETS="${CLEAN_CLUSTER_SUBNETS} --cluster-subnets=${i}"
# done

### NPF Alternative scaffolded by Gemini 3.1 Pro called using Github Copilot.
- It was asked, based on the NPF script, the cell that runs the NPF script, to create a python script that would perform the same thing, but without parsing the stdout as the test is going, and while also incorporating the CPU load monitoring.
- The code was thoroughly reviewed, and modified by hand

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import base64, csv, os, pathlib, re, shlex, time, subprocess
from dataclasses import dataclass
from typing import Literal
import enoslib as en
from datetime import datetime, timedelta
from fabric import Connection

# ---------- config ----------
RelayVersion = Literal["none", "RELAY", "APP_RELAY"]


@dataclass
class RunConfig:
    test_index: int  # 0 for latency test, 1 for segmentation test
    relay_version: RelayVersion
    additional_data_size: int
    lambda_: float
    test_length: int


@dataclass
class EvalConfig:
    n_runs: int = 3
    n_supplementary_runs: int = 2
    ready_sleep_relay: int = 1
    ready_sleep_clients: int = 2
    post_test_buffer: int = 3
    bin_log_level: str = "info"
    cert_path: str = "/tmp"
    server_bin: str = "/tmp/bin/server"
    client_bin: str = "/tmp/bin/client"
    fcquic_relay_bin: str = "/tmp/bin/fcquic_relay"
    app_relay_bin: str = "/tmp/bin/app_relay"
    remote_log_root: str = "/tmp/logs"
    local_out_dir: str = "./npf-out"
    num_ns_per_client: int = 5
    cc_algo: str = "disabled"
    fallback_delay: int = 10000
    interval: int = 100
    poisson: bool = True
    use_system_time: bool = True
    monitor_cpu: bool = True
    cpu_max: int = 1024
    server_cpus: str = "0-1"  # taskset -c range for server


# rates for 1100 bytes payloads -> lets say 1140 bytes
# 1140 * 8 = 9120, 9120*100 = 912000 bytes = 912 kbps
# lambda = 100 pkts/s -> 912kbps
# lambda = 1000 pkts/s -> 9.12mbps
# lambda = 10000 pkts/s -> 91.2 mbps
# lambda = 20000 pkts/s -> 182 mbps
# lambda = 50000 pkts/s -> 468 mbps


def latency_matrix():

    return [
        RunConfig(0, relay_version, 1100, lambda_=50000, test_length=10)
        for relay_version in ("none", "RELAY", "APP_RELAY")
    ]


def segmentation_matrix():
    return [
        RunConfig(1, relay_version, size, lambda_=20000, test_length=10)
        for relay_version in ("none", "RELAY", "APP_RELAY")
        for size in (1100, 2200)
    ]


# remote exec helpers
def create_fabric_conn(host, user="root") -> Connection:
    # Builds a fabric Connection
    host_addr = host.address if hasattr(host, "address") else host
    return Connection(host=host_addr, user=user)


def bg_inner_cmd(stdout, stderr, cmd):
    # Start cmd on each host with setsid so it survives SSH channel close
    return (
        f'mkdir -p "$(dirname {stdout})" "$(dirname {stderr})" && '
        f"setsid bash -c {shlex.quote(cmd)} > {stdout} 2> {stderr} < /dev/null &"
    )


def run_cmd_bg_enos(cmd, hosts, *, stdout, stderr, task_name="bg"):

    return en.run_command(
        bg_inner_cmd(stdout, stderr, cmd), roles=hosts, task_name=task_name
    )


def ssh_bg(cmd, host, *, stdout, stderr, user="root"):
    # Start command in the bg of the host via ssh
    conn = create_fabric_conn(host, user=user)
    conn.run(
        bg_inner_cmd(stdout, stderr, cmd), disown=True, hide=True, warn=True, pty=False
    )


# run a command synchronously no all given hosts in parallel
def run_cmd_ssh_parallel(cmd, hosts, *, check=True):
    def run_cmd_ssh_one_host(cmd, host, *, user="root", check=True):
        conn = create_fabric_conn(host, user=user)
        result = conn.run(cmd, hide=True, warn=not check, pty=False)

        if check and result.failed:
            raise RuntimeError(
                f"ssh to {conn.host} failed ({result.return_code}): {result.stderr.strip()}"
            )
        return result

    hosts = list(hosts)
    if not hosts:
        return []
    with ThreadPoolExecutor(max_workers=len(hosts)) as ex:
        return list(ex.map(lambda h: run_cmd_ssh_one_host(cmd, h, check=check), hosts))


# send a kill signal to the names of program on the given nodes
def send_pkill_hosts(hosts, names):
    joined = " ; ".join(f"pkill -9 {n} || true" for n in names)
    return run_cmd_ssh_parallel(joined, hosts, check=False)


# ---------- CPU load monitor (server only) ----------
# NOTE: code directly from NPF's cpuload.npf module, it has been modified to work in this context

# The script reads /proc/stat each second and output the following CSV columns: t_rel, cpu_id, util_pct (time relative to start, cpu id, utilization percentage)
# The rows where cpu_id is equal to -1 correspond to the mean across observed cores for that time sample
CPULOAD_SCRIPT = r"""
import sys, time
from collections import defaultdict
out_path, test_length = sys.argv[1], float(sys.argv[2])
cpu_min, cpu_max = int(sys.argv[3]), int(sys.argv[4])
last_idle, last_total = defaultdict(float), defaultdict(float)
start = time.time()
with open(out_path, "w", buffering=1) as out:
    out.write("t_rel,cpu_id,util_pct\n")
    first = True
    while time.time() - start < test_length:
        t_rel = time.time() - start
        rows, csum, ccnt = [], 0.0, 0
        with open("/proc/stat") as f:
            f.readline()  # skip aggregate cpu line
            for line in f:
                parts = line.strip().split()
                if not parts or not parts[0].startswith("cpu"):
                    break
                try:
                    cpuid = int(parts[0][3:])
                except ValueError:
                    continue
                vals = [float(x) for x in parts[1:]]
                idle, total = vals[3], sum(vals)
                di = idle - last_idle[cpuid]
                dt = total - last_total[cpuid]
                last_idle[cpuid], last_total[cpuid] = idle, total
                if first or dt <= 0:
                    continue
                util = 100.0 * (1.0 - di / dt)
                if cpu_min <= cpuid < cpu_max:
                    rows.append((cpuid, util))
                    csum += util
                    ccnt += 1
        if not first:
            for cid, u in rows:
                out.write("%.3f,%d,%.3f\n" % (t_rel, cid, u))
            if ccnt:
                out.write("%.3f,-1,%.3f\n" % (t_rel, csum / ccnt))
        first = False
        time.sleep(1)
"""
_CPULOAD_B64 = base64.b64encode(CPULOAD_SCRIPT.encode()).decode()


def install_cpuload(hosts, remote_path):
    cmd = f"echo {_CPULOAD_B64} | base64 -d > {remote_path}"
    return run_cmd_ssh_parallel(cmd, hosts)


def collect_cpuload(cfg, server_host, run_dir, test_name):
    local_dir = (
        pathlib.Path(cfg.local_out_dir)
        / "raw"
        / test_name
        / pathlib.Path(run_dir).name
        / "server"
    )
    local_dir.mkdir(parents=True, exist_ok=True)

    # download cpuload.csv
    conn = create_fabric_conn(server_host, user="root")
    remote_file = f"{run_dir}/server/cpuload.csv"
    local_file = str(local_dir / "cpuload.csv")
    try:
        conn.get(remote_file, local_file)
    except (FileNotFoundError, IOError):
        print(f"Failed to download cpuload.csv from {server_host}")

    samples = []
    csv_file = local_dir / "cpuload.csv"
    if csv_file.exists():
        with csv_file.open() as f:
            reader = csv.reader(f)
            next(reader, None)  # header
            for parts in reader:
                if len(parts) != 3:
                    continue
                try:
                    samples.append((float(parts[0]), int(parts[1]), float(parts[2])))
                except ValueError:
                    continue
    return samples


# ---------- command builders ----------
def server_cmd(cfg, rc, server_ip, run_dir):
    length = rc.test_length * 2
    qlog = f"{run_dir}/qlog/server"
    return (
        f"mkdir -p {qlog} && "
        f"env QLOGDIR={qlog} RUST_LOG_STYLE=never RUST_BACKTRACE=full "
        f"RUST_LOG={cfg.bin_log_level} taskset -c {cfg.server_cpus} {cfg.server_bin} "
        f"--cert-path {cfg.cert_path} --src {server_ip}:4433 --mc-src-addr {server_ip}:4443 "
        f"--test-mode --flexicast --fc-timer 0 --fall-back-delay {cfg.fallback_delay} "
        f"--unicast --fec-scheduler noredundancy --length {length} "
        f"--cc-algorithm {cfg.cc_algo} --fc-cwnd {cfg.cc_algo}"
    )


def relay_cmd(cfg, rc, server_ip, run_dir):
    bin_ = cfg.fcquic_relay_bin if rc.relay_version == "RELAY" else cfg.app_relay_bin
    length = rc.test_length * 2
    qlog = f"{run_dir}/qlog/relay"
    return (
        f"mkdir -p {qlog} && "
        f"CURRENT_RELAY_IP=$(ip -f inet addr show | grep inet | tail -1 | awk '{{print $2}}' | cut -d'/' -f1) && "
        f"env QLOGDIR={qlog} RUST_LOG_STYLE=never RUST_BACKTRACE=full "
        f"RUST_LOG={cfg.bin_log_level} {bin_} "
        f"https://{server_ip}:4433 --src $CURRENT_RELAY_IP:4433 --mc-src-addr $CURRENT_RELAY_IP:4443 "
        f"--cert-path {cfg.cert_path} --test-mode --flexicast --length {length} "
        f"--fc-timer 0 --fall-back-delay {cfg.fallback_delay} --unicast "
        f"--fec-scheduler noredundancy --cc-algorithm {cfg.cc_algo} --fc-cwnd {cfg.cc_algo}"
    )


def client_loop_cmd(cfg, rc, server_ip, relay_ips, run_dir, node_id, sleep_deadline_ts):
    relay_args = (
        " ".join(f"--relay-ips={ip}" for ip in relay_ips)
        if rc.relay_version != "none"
        else ""
    )
    poisson = f"--poisson --lambda {rc.lambda_}" if cfg.poisson else ""
    systime = "--use-system-time" if cfg.use_system_time else ""
    return f"""
mkdir -p {run_dir}/client
pids=()
for NS_IDX in $(seq $(( {cfg.num_ns_per_client} - 1 )) -1 0); do
    GLOBAL_IDX=$(( {node_id} * {cfg.num_ns_per_client} + NS_IDX ))
    CLIENT_ID=$(( GLOBAL_IDX + 1 ))
    NS_NAME="client-$NS_IDX"
    CLIENT_IP=$(ip netns exec $NS_NAME ip -f inet addr show | grep inet | tail -1 | awk '{{print $2}}' | cut -d'/' -f1)
    EXTRA=""; [ "$CLIENT_ID" = "1" ] && EXTRA="--sender"
    ip netns exec $NS_NAME env RUST_LOG_STYLE=never RUST_BACKTRACE=full RUST_LOG={cfg.bin_log_level} \\
        {cfg.client_bin} --server-ip {server_ip} --port 4433 {relay_args} \\
        -l $CLIENT_IP --flexicast -u CLIENT$CLIENT_ID --length {rc.test_length} --test-mode \\
        --conn-sleep-length 1 --test-start-ts {sleep_deadline_ts} --interval {cfg.interval} \\
        --additional-data-size {rc.additional_data_size} --cc-algorithm {cfg.cc_algo} \\
        --show-own-messages {poisson} {systime} $EXTRA \\
        > {run_dir}/client/client_$CLIENT_ID.stdout \\
        2> {run_dir}/client/client_$CLIENT_ID.stderr < /dev/null < /dev/null &
    pids+=($!)
done
for pid in "${{pids[@]}}"; do wait $pid; done
"""


def run_once(
    cfg,
    rc,
    roles_dict,
    node_ips,
    relay_ips,
    CLIENT_CLUSTERS,
    run_index,
    test_name,
):
    server_ip = node_ips["server"][0]
    run_id = f"run_t{rc.test_index}_{rc.relay_version}_sz{rc.additional_data_size}_r{run_index}"
    run_dir = f"{cfg.remote_log_root}/{test_name}/{run_id}"

    relay_hosts = [
        h for i in range(len(CLIENT_CLUSTERS)) for h in roles_dict[f"relay_{i}"]
    ]

    # make sure that each client is root because it has to start the clients in network namespaces
    client_hosts = [
        en.Host(h.address, alias=h.alias, user="root", extra=h.extra)
        for h in roles_dict["client"]
    ]
    all_hosts = roles_dict["server"] + client_hosts + relay_hosts

    # create the dirs on all of the hosts
    run_cmd_ssh_parallel(
        f"mkdir -p {run_dir}/server {run_dir}/relay {run_dir}/client {run_dir}/qlog",
        all_hosts,
    )

    # make sure all programs are stopped
    send_pkill_hosts(all_hosts, ["server", "fcquic_relay", "app_relay", "client"])
    time.sleep(1)

    # start server in bg
    run_cmd_bg_enos(
        server_cmd(cfg, rc, server_ip, run_dir),
        roles_dict["server"],
        stdout=f"{run_dir}/server/server.stdout",
        stderr=f"{run_dir}/server/server.stderr",
        task_name="server",
    )

    # start CPU load monitor (server only)
    if cfg.monitor_cpu:
        cpuload_py = f"{run_dir}/server/cpuload.py"
        cpuload_csv = f"{run_dir}/server/cpuload.csv"
        install_cpuload(roles_dict["server"], cpuload_py)

        run_cmd_bg_enos(
            f"python3 -u {cpuload_py} {cpuload_csv} {rc.test_length + 5} 0 {cfg.cpu_max}",
            roles_dict["server"],
            stdout=f"{run_dir}/server/cpuload.stdout",
            stderr=f"{run_dir}/server/cpuload.stderr",
            task_name="start_cpuload",
        )

    time.sleep(cfg.ready_sleep_relay)

    # start relays if we're testing with them
    if rc.relay_version != "none":
        run_cmd_bg_enos(
            relay_cmd(cfg, rc, server_ip, run_dir),
            relay_hosts,
            stdout=f"{run_dir}/relay/relay_$(hostname).stdout",
            stderr=f"{run_dir}/relay/relay_$(hostname).stderr",
            task_name="relay",
        )

    # pick a timestamp in 5 seconds, we pass this to all of the clients that will all wait until that timestamp is reached before starting
    datetime_now = datetime.now()
    sleep_deadline = datetime_now + timedelta(seconds=5)
    sleep_deadline_ts = sleep_deadline.timestamp()

    # start all clients in // to make them start kinda at the same time
    def _start_client(node_id, h):
        cmd = client_loop_cmd(
            cfg, rc, server_ip, relay_ips, run_dir, node_id, sleep_deadline_ts
        )
        ssh_bg(
            cmd,
            h,
            stdout=f"{run_dir}/client/loop_{node_id}.stdout",
            stderr=f"{run_dir}/client/loop_{node_id}.stderr",
        )

    with ThreadPoolExecutor(max_workers=max(1, len(client_hosts))) as ex:
        list(ex.map(lambda p: _start_client(*p), list(enumerate(client_hosts))))

    # wait for test duration to pass
    time.sleep(rc.test_length + cfg.post_test_buffer)

    send_pkill_hosts(all_hosts, ["server", "fcquic_relay", "app_relay", "client"])

    time.sleep(1)

    latencies = collect_latencies(cfg, client_hosts, run_dir, test_name)
    cpu_samples = (
        collect_cpuload(cfg, roles_dict["server"][0], run_dir, test_name)
        if cfg.monitor_cpu
        else []
    )
    return latencies, cpu_samples


# ---------- Logs collection ----------
RESULT_RE = re.compile(r"^RESULT-LATENCY\s+([0-9.]+)\s*$")


def download_run_logs_host(host, run_dir, local_root):
    host_dir = local_root / host.alias
    host_dir.mkdir(exist_ok=True)

    subprocess.run(
        [
            "rsync",
            "-az",
            "-e",
            "ssh -o StrictHostKeyChecking=no -o BatchMode=yes -o LogLevel=ERROR",
            "--include=*/",
            "--include=client_*.stdout",
            "--exclude=*",
            f"root@{host.address}:{run_dir}/client/",
            f"{host_dir}/",
        ],
        check=False,
    )
    return host_dir


def collect_latencies(cfg, client_hosts, run_dir, test_name):
    local_root = (
        pathlib.Path(cfg.local_out_dir) / "raw" / test_name / pathlib.Path(run_dir).name
    )
    local_root.mkdir(parents=True, exist_ok=True)

    # pull all hosts in parallel
    with ThreadPoolExecutor(max_workers=min(4, max(1, len(client_hosts)))) as ex:
        host_dirs = list(
            ex.map(
                lambda h: download_run_logs_host(h, run_dir, local_root), client_hosts
            )
        )

    latencies = []
    for host_dir in host_dirs:
        for f in host_dir.rglob("client_*.stdout"):
            for line in f.read_text(errors="replace").splitlines():
                m = RESULT_RE.match(line.strip())
                if m:
                    latencies.append(float(m.group(1)))
    return latencies


def write_csv(path, rows, fieldnames):
    print("Writing CSV file...")
    with path.open("w", newline="") as f:
        w = csv.DictWriter(
            f,
            fieldnames=fieldnames,
            quoting=csv.QUOTE_NONNUMERIC,
        )
        w.writeheader()
        w.writerows(rows)
    print(f"wrote {path} ({len(rows)} rows)")


# ---------- driver ----------
def run_eval(matrix, cfg, roles_dict, node_ips, relay_ips, CLIENT_CLUSTERS, test_name):
    start = time.time()
    lat_path = pathlib.Path(cfg.local_out_dir) / f"{test_name}.csv"
    cpu_path = pathlib.Path(cfg.local_out_dir) / f"{test_name}_cpu.csv"

    lat_path.parent.mkdir(parents=True, exist_ok=True)
    lat_rows, cpu_rows = [], []
    global_idx, cpu_idx = 0, 0

    for rc in matrix:
        successful = 0
        for attempt in range(cfg.n_runs + cfg.n_supplementary_runs):
            if successful >= cfg.n_runs:
                break

            print(
                f"=> test={rc.test_index} relay={rc.relay_version} "
                f"size={rc.additional_data_size} run={successful} (attempt {attempt + 1})"
            )

            # run the test
            try:
                lats, cpu = run_once(
                    cfg,
                    rc,
                    roles_dict,
                    node_ips,
                    relay_ips,
                    CLIENT_CLUSTERS,
                    successful,
                    test_name,
                )
            except Exception as e:
                print(f"failed: {e}")
                continue

            if not lats:
                print("no RESULT-LATENCY lines, retrying")
                continue

            print(f"-> collected {len(lats)} latencies, {len(cpu)} cpu samples\n")
            for y in lats:
                lat_rows.append(
                    {
                        "index": global_idx,
                        "test_index": rc.test_index,
                        "ADDITIONAL_DATA_SIZE": rc.additional_data_size,
                        "RELAY_VERSION": f'"{rc.relay_version}"',
                        "y_LATENCY": y,
                        "run_index": successful,
                    }
                )
                global_idx += 1
            for time_relative, cpu_id, utilization in cpu:
                cpu_rows.append(
                    {
                        "index": cpu_idx,
                        "test_index": rc.test_index,
                        "ADDITIONAL_DATA_SIZE": rc.additional_data_size,
                        "RELAY_VERSION": f'"{rc.relay_version}"',
                        "run_index": successful,
                        "time_rel": time_relative,
                        "cpu_id": cpu_id,
                        "utilization_percentage": utilization,
                    }
                )
                cpu_idx += 1
            successful += 1

    elapsed = time.time() - start
    print(f"\nTest finished in {elapsed} seconds")
    write_csv(
        lat_path,
        fieldnames=[
            "index",
            "test_index",
            "ADDITIONAL_DATA_SIZE",
            "RELAY_VERSION",
            "y_LATENCY",
            "run_index",
        ],
        rows=lat_rows,
    )

    if cpu_rows:
        write_csv(
            cpu_path,
            fieldnames=[
                "index",
                "test_index",
                "ADDITIONAL_DATA_SIZE",
                "RELAY_VERSION",
                "run_index",
                "time_rel",
                "cpu_id",
                "utilization_percentage",
            ],
            rows=cpu_rows,
        )

    return lat_path


# ---------- launch ----------
LATENCY_TEST = True
N_RUNS = 5

cfg = EvalConfig(
    n_runs=N_RUNS, cpu_max=2, num_ns_per_client=5
)  # monitor cores 0-2 exclusive, matching server_cpus
relay_ips = [node_ips[f"relay_{i}"][0] for i in range(len(CLIENT_CLUSTERS))]
now = datetime.now().strftime("%d-%m-%H-%M%p")


test_prefix = "serv_2thr_"
test_name = ""
matrix = None

if LATENCY_TEST:
    test_name = f"{test_prefix}latency_test_large_relay_topo_{now}"
    matrix = latency_matrix()
else:
    test_name = f"{test_prefix}segmentation_test_large_relay_topo_{now}"
    matrix = segmentation_matrix()

run_eval(
    matrix,
    cfg,
    roles,
    node_ips,
    relay_ips,
    CLIENT_CLUSTERS,
    test_name=test_name,
)

#### Downloading SQLOGs from server and relay

In [ ]:
import subprocess
import os
from pathlib import Path

remote_bin_dir = "/tmp"
remote_log_root = "/tmp/logs"

if LATENCY_TEST:
    matrix = latency_matrix()
else:
    matrix = segmentation_matrix()

local_base = Path(f"./sqlogs/{test_name}")

for run_conf in matrix:
    for run_index in range(N_RUNS):
        run_id = f"run_t{run_conf.test_index}_{run_conf.relay_version}_sz{run_conf.additional_data_size}_r{run_index}"

        remote_qlog_dir = f"{remote_log_root}/{test_name}/{run_id}/qlog/server"
        local_dir = local_base / run_id / "server"
        local_dir.mkdir(parents=True, exist_ok=True)

        # download sqlogs from server only
        for node in roles["server"]:
            host = node.address
            print(f"downloading sqlogs from {host} to {remote_qlog_dir}")
            subprocess.run(
                [
                    "rsync",
                    "-az",
                    "-o LogLevel=ERROR",
                    "--include=*.sqlog",
                    "--exclude=*",
                    f"root@{host}:{remote_qlog_dir}/",
                    f"{local_dir}/",
                ],
                check=False,
            )

print(f"results: {local_base}")

### Merging SQLOG files together

In [ ]:
from pathlib import Path
import sys
import tempfile
import re
import json
import csv


def merge_sqlogs(files, output):
    control_chars = re.compile(r"[\x00-\x08\x0b-\x1f\x7f]")

    with open(output, "w") as out:
        for i, f in enumerate(files):
            with open(f) as src:
                first = True
                for j, line in enumerate(src):
                    if j == 0 and i > 0:
                        # if i > 0, then we wrote the header once already, so now skip the headers (first lines of sqlog files: j==0)
                        continue

                    line = line.rstrip() + "\n"
                    line = control_chars.sub("", line)
                    out.write(control_chars.sub("", line))


def extract_path_acks(sqlog, csv_out):

    # we need to get the path_ack lengths from the sqlogs
    # go through the merged sqlog, and append to a csv file the time and length of each path_ack we see
    with open(sqlog) as src, open(csv_out, "w", newline="") as out:
        writer = csv.writer(out)
        writer.writerow(["time", "length"])

        for line in src:
            line = line.strip()
            if not line:
                continue

            try:
                event = json.loads(line)
            except json.JSONDecodeError:
                # idk why so many log entries are broken
                continue

            # e.g.
            # {"time":12.632589,"name":"transport:packet_received","data":{"header":{"packet_type":"1RTT","packet_number":4},"raw":{"length":1155,"payload_length":1138},"frames":[{"frame_type":"path_ack","path_identifier":0,"ack_delay":0.085,"acked_ranges":[[3,3]]},{"frame_type":"path_new_connection_id","path_id":1,"sequence_number":0,"retire_prior_to":0,"connection_id_length":16,"connection_id":"ab44f5dde5157072f203e53c8d76836d","stateless_reset_token":"536abfd4dc2107e6e1188cbba08f4ce1"},{"frame_type":"padding","payload_length":1071}]}}
            if event.get("name") == "transport:packet_received":
                frames = event.get("data", {}).get("frames", []) or []

                for frame in frames:
                    if frame.get("frame_type") == "path_ack":
                        # if we do have a path_ack frame (migth contain other stuff), get the length
                        length = event.get("data", {}).get("raw", {}).get("length")

                        if length is not None:
                            writer.writerow([event.get("time"), length])


local_base = Path(f"./sqlogs/{test_name}")

for relay_test in ["none", "RELAY", "APP_RELAY"]:

    trace_files = []
    for run_conf in matrix:
        if run_conf.relay_version != relay_test:
            continue

        for run_index in range(N_RUNS):
            run_id = f"run_t{run_conf.test_index}_{run_conf.relay_version}_sz{run_conf.additional_data_size}_r{run_index}"

            server_dir = local_base / run_id / "server"

            for file in sorted(server_dir.glob("server-server-*.sqlog")):
                trace_files.append(file)

    if not trace_files:
        print(f"no files found for {relay_test}")
        continue

    print(f"relay={relay_test}: {len(trace_files)} trace files")

    temp_dir = Path(tempfile.mkdtemp(prefix="ackrate_"))
    merged_log = temp_dir / f"merged_{relay_test}.sqlog"

    merge_sqlogs(trace_files, merged_log)

    merged_csv = Path(f"./npf-out/ack_rate_{test_name}") / f"{relay_test}.csv"
    merged_csv.parent.mkdir(parents=True, exist_ok=True)

    extract_path_acks(merged_log, merged_csv)
    print(f"path_ack csv: {merged_csv}")

### Graphing the results

In [3]:
import subprocess
from pathlib import Path

INSET_GRAPHS = True
NO_TITLE = True
out_path = f"./graphs/{test_name}/"
output_path = Path(out_path)
output_path.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        "./relay_graphs.py",
        f"./npf-out/{test_name}.csv",  # input csv paht
        out_path,  # out path
        test_name,
        f"./npf-out/ack_rate_{test_name}/",  # ack_rate_path
        f"./npf-out/{test_name}_cpu.csv",  # cpu_csv_path
        *(["--inset"] if INSET_GRAPHS else []),
        *(
            ["--no-title"] if NO_TITLE else []
        ),  # list unpacking, this avoids the empty ""
    ],
    check=True,
)

ADDITIONAL_DATA_SIZE values: [np.int64(1100)]
No relay samples: 263731
FCQUIC relay samples: 163251
APP relay samples: 276734
Per run breakdown
none:
  Run 0: 263731 samples
RELAY:
  Run 0: 163251 samples
APP_RELAY:
  Run 0: 276734 samples
min length of the dataframes: 163251
only one additional data size, skipping mean/median plot.


CompletedProcess(args=['./relay_graphs.py', './npf-out/serv_2thr_latency_test_large_relay_topo_26-04-21-39PM.csv', './graphs/serv_2thr_latency_test_large_relay_topo_26-04-21-39PM/', 'serv_2thr_latency_test_large_relay_topo_26-04-21-39PM', './npf-out/ack_rate_serv_2thr_latency_test_large_relay_topo_26-04-21-39PM/', './npf-out/serv_2thr_latency_test_large_relay_topo_26-04-21-39PM_cpu.csv', '--inset', '--no-title'], returncode=0)

### Compress csv results to be able to push to github
And then delete the csv file

In [ ]:
import subprocess

# compress all related files in one tarball
subprocess.run(
    [
        "tar",
        "czf",
        f"./npf-out/all_{test_name}.tar.gz",
        f"./npf-out/{test_name}.csv",
        f"./npf-out/{test_name}_cpu.csv",
        f"./npf-out/ack_rate_{test_name}/",
        f"./npf-out/raw/{test_name}/",
        f"./sqlogs/{test_name}/",
    ],
    check=True,
)

# move archive to the graph dir of the test
subprocess.run(
    [
        "mv",
        f"./npf-out/all_{test_name}.tar.gz",
        f"./graphs/{test_name}/{test_name}.tar.gz",
    ],
    check=True,
)

# delete the csvs and directories
subprocess.run(
    [
        "rm",
        f"./npf-out/{test_name}.csv",
        f"./npf-out/{test_name}_cpu.csv",
    ],
    check=True,
)
subprocess.run(
    [
        "rm",
        "-rf",
        f"./npf-out/ack_rate_{test_name}/",
        f"./npf-out/raw/{test_name}/",
        f"./sqlogs/{test_name}/",
    ],
    check=True,
)

##### Unarchive tarball to re create graphs

In [2]:
import subprocess
from pathlib import Path

# og_name = "sserv_2thr_latency_test_large_relay_topo_26-04-21-39PM_481mbps"
test_name = "serv_2thr_latency_test_large_relay_topo_26-04-21-39PM"

archive_path = Path(f"./graphs/{test_name}/{test_name}.tar.gz")

if archive_path.exists():
    print(f"decompressing {archive_path}...")

    Path("./npf-out/").mkdir(parents=True, exist_ok=True)

    subprocess.run(
        [
            "tar",
            "xzf",
            str(archive_path),
            "-C",
            "./",
        ],
        check=True,
    )
    print(f"decompressed files to ./npf-out/ and ./sqlogs/")
else:
    print(f"Couldn't find: {archive_path}")

decompressing graphs/serv_2thr_latency_test_large_relay_topo_26-04-21-39PM/serv_2thr_latency_test_large_relay_topo_26-04-21-39PM.tar.gz...
decompressed files to ./npf-out/ and ./sqlogs/


#### Deleting log files from all clusters

In [ ]:
import subprocess
from pathlib import Path

remote_log_root = "/tmp/logs"

if LATENCY_TEST:
    matrix = latency_matrix()
else:
    matrix = segmentation_matrix()

for run_conf in matrix:

    remote_qlog_dir = f"{remote_log_root}/{test_name}/"

    en.run_command(
        f"rm -rf {remote_qlog_dir}",
        roles=roles["client"] + roles["relay"] + roles["server"],
    )

print("done deleting sqlog files")

### Stopping the current booking

In [ ]:
provider.destroy()